In [ ]:
%pip install -q implicit lightgbm xgboost catboost

In [ ]:
import gc
import pickle
import numpy as np
import pandas as pd
import polars as pl
from datetime import datetime
from scipy.sparse import csr_matrix
from scipy import sparse
import implicit
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRanker, Pool
from collections import defaultdict
import random
import pickle
import os
import time

In [ ]:
# Cấu hình
pl.Config.set_tbl_rows(20)
RANDOM_STATE = 42

def clean_ram():
    gc.collect()

In [ ]:
# 1. LOAD DATA & MERGE
print("\nLoading Data...")
data_path = '/kaggle/input/datasetcs116/transaction_2024'
gt_path = '/kaggle/input/2025gt/final_groundtruth.pkl'

# 1.1. Load Ground Truth
with open(gt_path, 'rb') as f:
    gt_data = pickle.load(f)

ground_truth_dict = {}
if isinstance(gt_data, pd.DataFrame):
    gt_data['customer_id'] = gt_data['customer_id'].astype(str)
    first_item = gt_data['item_id'].iloc[0]
    if isinstance(first_item, (list, np.ndarray)):
        ground_truth_dict = dict(zip(gt_data['customer_id'], gt_data['item_id']))
    else:
        gt_data['item_id'] = gt_data['item_id'].astype(str)
        ground_truth_dict = gt_data.groupby('customer_id')['item_id'].apply(list).to_dict()
else:
    ground_truth_dict = {str(k): [str(i) for i in (v if isinstance(v, list) else [v])]
                          for k, v in gt_data.items()}

target_customers = set(ground_truth_dict.keys())
print(f"   Total GT users: {len(target_customers):,}")

# 1.2. Load Transactions
# Load 2024
df_trans_2024 = pl.read_parquet(f'{data_path}/sales_pers.purchase_history_daily_chunk_*.parquet')
df_trans_2024 = df_trans_2024.rename({
    df_trans_2024.columns[0]: 'timestamp',
    df_trans_2024.columns[1]: 'user_id',
    df_trans_2024.columns[2]: 'item_id'
}).select([
    pl.col('user_id').cast(pl.String),
    pl.col('item_id').cast(pl.String),
    (pl.col('timestamp') * 1000).cast(pl.Datetime("ms")).alias('timestamp')
])

# Load 2025
pkl_path = '/kaggle/input/2025gt/01-2025.pkl'
df_2025_pd = pd.read_pickle(pkl_path)
df_trans_2025 = pl.from_pandas(df_2025_pd[['user_id', 'item_id', 'timestamp']]).select([
    pl.col('user_id').cast(pl.String),
    pl.col('item_id').cast(pl.String),
    (pl.col('timestamp') * 1000).cast(pl.Datetime("ms")).alias('timestamp')
])
del df_2025_pd
clean_ram()

# Merge
df_trans = pl.concat([df_trans_2024, df_trans_2025]).sort("timestamp")
del df_trans_2024, df_trans_2025
clean_ram()

# 1.3. Mapping User
df_user = pl.read_parquet(f'{data_path}/sales_pers.user_chunk_*.parquet')
df_user = df_user.rename({
    df_user.columns[0]: 'customer_id',
    df_user.columns[-2]: 'user_id'
}).select([
    pl.col('user_id').cast(pl.String),
    pl.col('customer_id').cast(pl.String)
]).unique(subset=['user_id'])

pdf_user = df_user.to_pandas()
cust_to_user = dict(zip(pdf_user['customer_id'], pdf_user['user_id']))
user_to_cust = dict(zip(pdf_user['user_id'], pdf_user['customer_id']))

target_user_ids = set()
for cid in target_customers:
    if cid in cust_to_user:
        target_user_ids.add(cust_to_user[cid])

print(f"   Target User IDs found: {len(target_user_ids):,}")

In [ ]:
# 2. NEW FEATURES FOR COLD ITEMS (TRENDING ITEMS)
print("\n>>> [Feature Engineering] Calculating Global Trends...")
REF_DATE = datetime(2025, 1, 31)

# 2.1 Calculate item popularity in 7 and 30 days (for cold items)
df_trans_calc = df_trans.with_columns([
    (REF_DATE - pl.col("timestamp")).dt.total_days().alias("days_ago")
])

# 2.2 Global Popularity 7d
global_pop_7d = dict(df_trans_calc.filter(pl.col("days_ago") <= 7)
                     .group_by("item_id").len().rename({"len": "pop_7d"})
                     .iter_rows())

# 2.3 Global Popularity 30d
global_pop_30d = dict(df_trans_calc.filter(pl.col("days_ago") <= 30)
                     .group_by("item_id").len().rename({"len": "pop_30d"})
                     .iter_rows())

# 2.4 Global All time
item_popularity = dict(df_trans.group_by('item_id').len().iter_rows())

# 2.5 Get Top Items (Trending items) to fill in candidates
global_top_items = [x[0] for x in df_trans_calc.filter(pl.col("days_ago") <= 30)
                    .group_by('item_id').len().sort('len', descending=True)
                    .head(200).iter_rows()]

print("   ✅ Global Trends Calculated.")

# 2.6 Advanced User-Item Features
def build_advanced_features(df, target_uids):
    # Get user to train + target users
    all_users = df['user_id'].unique().to_list()
    train_users = set(random.sample(all_users, min(20000, len(all_users))))
    relevant_users = target_uids.union(train_users)

    df_filtered = df.filter(pl.col("user_id").is_in(list(relevant_users)))

    # Calculate cycle features
    df_calc = df_filtered.with_columns([
        (REF_DATE - pl.col("timestamp")).dt.total_days().alias("days_ago")
    ])

    df_features = df_calc.select(["user_id", "item_id"]).unique()

    # Cycles & Last Buy
    df_cycles = (df_calc.sort(["user_id", "item_id", "timestamp"])
                 .with_columns([
                     pl.col("timestamp").diff().over(["user_id", "item_id"]).dt.total_days().alias("diff_days")
                 ]))

    cycle_stats = (df_cycles.group_by(["user_id", "item_id"])
                   .agg([
                       pl.col("diff_days").mean().alias("avg_cycle"),
                       pl.col("days_ago").min().alias("last_buy_days"),
                       pl.col("days_ago").count().alias("total_cnt") # Thêm tổng số lần mua
                   ]))

    df_features = df_features.join(cycle_stats, on=["user_id", "item_id"], how="left")

    print("   Converting History Features to Dictionary...")
    adv_feats_dict = {}
    for row in df_features.iter_rows(named=True):
        uid = str(row['user_id'])
        iid = str(row['item_id'])
        if uid not in adv_feats_dict: adv_feats_dict[uid] = {}

        avg_cycle = row['avg_cycle'] if row['avg_cycle'] is not None else 999
        last_buy = row['last_buy_days']
        overdue = last_buy - avg_cycle if row['avg_cycle'] is not None else -999

        adv_feats_dict[uid][iid] = {
            'cnt': int(row['total_cnt']),
            'cyc': float(avg_cycle),
            'ovr': float(overdue),
            'last': float(last_buy)
        }
    return adv_feats_dict

adv_feats_dict = build_advanced_features(df_trans, target_user_ids)
clean_ram()

In [ ]:
# 3. FREQUENCY & ALS
print("\n Building Frequency & ALS...")

# 3.1 Frequency Dicts for fast lookup
user_item_frequency = defaultdict(lambda: defaultdict(int))
for row in df_trans.select(['user_id', 'item_id']).iter_rows():
    user_item_frequency[row[0]][row[1]] += 1

customer_item_frequency = defaultdict(lambda: defaultdict(int))
for uid, item_freq in user_item_frequency.items():
    cid = user_to_cust.get(uid)
    if cid:
        customer_item_frequency[cid] = item_freq

# 3.2 ALS Training
unique_users = df_trans['user_id'].unique().to_list()
unique_items = df_trans['item_id'].unique().to_list()
user_to_idx = {v: k for k, v in enumerate(unique_users)}
idx_to_item = {k: v for k, v in enumerate(unique_items)}
item_to_idx = {v: k for k, v in enumerate(unique_items)}

train_df = df_trans.with_columns([
    pl.col("user_id").replace_strict(user_to_idx, default=None).cast(pl.Int32).alias("u_idx"),
    pl.col("item_id").replace_strict(item_to_idx, default=None).cast(pl.Int32).alias("i_idx")
]).drop_nulls()

counts = train_df.group_by(['u_idx', 'i_idx']).len()
matrix = csr_matrix((counts['len'].to_numpy().astype(np.float32),
                    (counts['u_idx'].to_numpy(), counts['i_idx'].to_numpy())),
                    shape=(len(unique_users), len(unique_items)))

model_als = implicit.als.AlternatingLeastSquares(factors=128, regularization=0.05, iterations=15, random_state=42)
model_als.fit(matrix)
user_items_csr = matrix.tocsr()
print("   ✅ ALS trained")

# 3.3 Item Similarity (Top items that usually go together)
item_matrix = matrix.T.tocsr()
norms = sparse.linalg.norm(item_matrix, axis=1)
norms[norms == 0] = 1
item_matrix = item_matrix.multiply(1.0 / norms.reshape(-1, 1)).tocsr()
sim_matrix = item_matrix @ item_matrix.T
item_similarity = {}
for i in range(sim_matrix.shape[0]):
    row = sim_matrix[i].toarray().flatten()
    row[i] = 0
    # Lấy top 10 tương đồng
    top_k = np.argsort(row)[::-1][:10]
    item_similarity[i] = [int(j) for j in top_k if row[j] > 0.05]
del item_matrix, sim_matrix, norms
clean_ram()

In [ ]:
# 4. BUILD TRAINING DATA
print("\n Building Training Data...")
sample_cids = random.sample(list(customer_item_frequency.keys()), min(15000, len(customer_item_frequency)))

ltr_rows, ltr_labels, ltr_groups = [], [], []

for cid in sample_cids:
    gt_items = set(str(i) for i in ground_truth_dict.get(cid, []))
    if not gt_items: continue
    uid = cust_to_user.get(cid)
    u_idx = user_to_idx.get(uid)
    if u_idx is None: continue

    # --- Candidates ---
    candidates = set()
    # Repurchase (High priority)
    candidates.update(customer_item_frequency[cid].keys())

    # ALS (Find new items)
    try:
        ids, scores = model_als.recommend([u_idx], user_items_csr[[u_idx]], N=50, filter_already_liked_items=False)
        for i in ids[0]: candidates.add(idx_to_item[i])
    except: pass

    # --- Feature Extraction ---
    current_items = []
    for item_id in list(candidates)[:80]: # Limit
        # Label
        label = 1 if item_id in gt_items else 0

        # Features
        # 1. History Features
        adv = adv_feats_dict.get(uid, {}).get(item_id, {})
        hist_cnt = adv.get('cnt', 0)
        hist_cyc = adv.get('cyc', 999)
        hist_ovr = adv.get('ovr', -999)
        hist_last = adv.get('last', 999)

        # 2. Global Trend Features
        pop_all = item_popularity.get(item_id, 0)
        pop_7d = global_pop_7d.get(item_id, 0)
        pop_30d = global_pop_30d.get(item_id, 0)

        # Feature Vector: [Cnt, Cyc, Ovr, Last, PopAll, Pop7d, Pop30d]
        feats = [hist_cnt, hist_cyc, hist_ovr, hist_last, pop_all, pop_7d, pop_30d]

        current_items.append(feats)
        ltr_labels.append(label)

    if current_items:
        ltr_rows.extend(current_items)
        ltr_groups.append(len(current_items))

X_train = np.array(ltr_rows)
y_train = np.array(ltr_labels)
print(f"   Train Size: {len(X_train):,}")
clean_ram()

In [ ]:
# 5. TRAIN MODELS
print("\n Training Models...")

# 1. LightGBM
print("   Training LightGBM...")
train_data = lgb.Dataset(X_train, label=y_train, group=ltr_groups)
params = {
    'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_at': [10],
    'learning_rate': 0.05, 'num_leaves': 63, 'verbose': -1,
    'device': 'gpu', 'gpu_device_id': 0, 'gpu_platform_id': 0
}
try:
    model_lgb = lgb.train(params, train_data, num_boost_round=300)
except:
    params['device'] = 'cpu'
    model_lgb = lgb.train(params, train_data, num_boost_round=300)
clean_ram()

# 2. XGBoost
print("   Training XGBoost...")
model_xgb = xgb.XGBRanker(
    objective='rank:ndcg', eval_metric='ndcg@10', learning_rate=0.05,
    n_estimators=300, max_depth=8, tree_method='hist', device='cuda:0'
)
try:
    model_xgb.fit(X_train, y_train, group=ltr_groups, verbose=False)
except:
    model_xgb = xgb.XGBRanker(
        objective='rank:ndcg', eval_metric='ndcg@10', learning_rate=0.05,
        n_estimators=300, max_depth=8, tree_method='hist', device='cpu'
    )
    model_xgb.fit(X_train, y_train, group=ltr_groups, verbose=False)
clean_ram()

In [ ]:
# 6. PREDICTION
print("\n Prediction (W-history and W/o-History)...")
pred_with_hist = {}
pred_no_hist = {}

BATCH_SIZE = 2000
keys = list(ground_truth_dict.keys())

for i in range(0, len(keys), BATCH_SIZE):
    if i % 10000 == 0: print(f"   Batch {i}...")
    batch_cids = keys[i : i + BATCH_SIZE]

    for cid in batch_cids:
        uid = cust_to_user.get(cid)
        u_idx = user_to_idx.get(uid) if uid else None

        # 6.1. Identify Items Bought 
        bought_items = set(customer_item_frequency.get(cid, {}).keys())

        # 6.2. Candidate Generation 
        candidates = set()
        candidates.update(bought_items) # Repurchase

        # ALS
        if u_idx is not None:
            try:
                ids, _ = model_als.recommend([u_idx], user_items_csr[[u_idx]], N=100, filter_already_liked_items=False)
                for x in ids[0]: candidates.add(idx_to_item[x])
            except: pass

            # Item Similarity
            recent_bought = list(bought_items)[:5]
            for item_b in recent_bought:
                b_idx = item_to_idx.get(item_b)
                if b_idx in item_similarity:
                    for s_idx in item_similarity[b_idx]:
                        candidates.add(idx_to_item[s_idx])
        
        non_bought_count = sum(1 for c in candidates if c not in bought_items)
        if non_bought_count < 20:
            for top_item in global_top_items[:50]:
                candidates.add(top_item)

        final_cands = list(candidates)
        if not final_cands:
            pred_with_hist[cid] = global_top_items[:10]
            pred_no_hist[cid] = [x for x in global_top_items if x not in bought_items][:10]
            continue

        # 6.3. Build Feature Matrix 
        X_pred = []
        for item_id in final_cands:
            adv = adv_feats_dict.get(uid, {}).get(item_id, {})
            # Features phải khớp với lúc train: [Cnt, Cyc, Ovr, Last, PopAll, Pop7d, Pop30d]
            feats = [
                adv.get('cnt', 0), adv.get('cyc', 999), adv.get('ovr', -999), adv.get('last', 999),
                item_popularity.get(item_id, 0),
                global_pop_7d.get(item_id, 0),
                global_pop_30d.get(item_id, 0)
            ]
            X_pred.append(feats)

        X_pred = np.array(X_pred)

        # 6.4. Scoring 
        p1 = model_lgb.predict(X_pred)
        p2 = model_xgb.predict(X_pred)
        avg_score = 0.6*p1 + 0.4*p2

        # Create list (item, score)
        item_scores = list(zip(final_cands, avg_score))
        item_scores.sort(key=lambda x: x[1], reverse=True)

        # 6.5. Generate Lists
        # List A: with history
        pred_with_hist[cid] = [x[0] for x in item_scores[:10]]

        # List B: without History
        no_hist_list = []
        for item, score in item_scores:
            if item not in bought_items:
                no_hist_list.append(item)
                if len(no_hist_list) == 10: break

        if len(no_hist_list) < 10:
            for top in global_top_items:
                if top not in bought_items and top not in no_hist_list:
                    no_hist_list.append(top)
                    if len(no_hist_list) == 10: break

        pred_no_hist[cid] = no_hist_list

In [ ]:
# 7. EVAL & SAVE
def precision_at_k(pred, gt, K=10):
    precisions = []
    for user, gt_items in gt.items():
        if user not in pred: continue
        relevant = set(gt_items)
        p_items = pred[user][:K]
        hits = len(set(p_items) & relevant)
        precisions.append(hits/K)
    return np.mean(precisions) if precisions else 0

score_hist = precision_at_k({k:[str(i) for i in v] for k,v in pred_with_hist.items()},
                            {k:[str(i) for i in v] for k,v in ground_truth_dict.items()})
print(f"\nFINAL SCORE (With History): {score_hist:.6f}")

print("\n SAVING FILES...")

# Save With History
with open('prediction.pkl', 'wb') as f:
    pickle.dump(pred_with_hist, f)
print("Saved 'prediction.pkl'")

# Save No History
with open('prediction_no_hist_dict.pkl', 'wb') as f:
    pickle.dump(pred_no_hist, f)
print("Saved 'prediction_no_hist_dict.pkl'")

size_mb = os.path.getsize('prediction_no_hist_dict.pkl') / (1024 * 1024)
print(f"   ℹ️ Prediction No History Size: {size_mb:.2f} MB (Should be large ~60-70MB)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


🚀 HYBRID RECSYS: REPURCHASE + COLD START IMPROVEMENT

>>> [1/8] Loading Data...


/tmp/ipykernel_55/314532615.py:44: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  gt_data = pickle.load(f)


   Total GT users: 644,970
   Target User IDs found: 644,970

>>> [Feature Engineering] Calculating Global Trends...
   ✅ Global Trends Calculated.
   Converting History Features to Dictionary...

>>> [2/8] Building Frequency & ALS...


...

   ✅ ALS trained

>>> [5/8] Building Training Data...
   Train Size: 180,288

>>> [6/8] Training Models...
   Training LightGBM...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


   Training XGBoost...

>>> [7/8] Prediction (Optimized for No-History)...
   Batch 0...


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [01:02:02] WARNING: /workspace/src/common/error_msg.cc:41: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


   Batch 10000...
   Batch 20000...
   Batch 30000...
   Batch 40000...
   Batch 50000...
   Batch 60000...
   Batch 70000...
   Batch 80000...
   Batch 90000...
   Batch 100000...
   Batch 110000...
   Batch 120000...
   Batch 130000...
   Batch 140000...
   Batch 150000...
   Batch 160000...
   Batch 170000...
   Batch 180000...
   Batch 190000...
   Batch 200000...
   Batch 210000...
   Batch 220000...
   Batch 230000...
   Batch 240000...
   Batch 250000...
   Batch 260000...
   Batch 270000...
   Batch 280000...
   Batch 290000...
   Batch 300000...
   Batch 310000...
   Batch 320000...
   Batch 330000...
   Batch 340000...
   Batch 350000...
   Batch 360000...
   Batch 370000...
   Batch 380000...
   Batch 390000...
   Batch 400000...
   Batch 410000...


In [ ]:
import pickle
print(f"Đang lưu {len(pred_with_hist)} users đã xử lý...")
with open('prediction_checkpoint_410k.pkl', 'wb') as f:
    pickle.dump(pred_with_hist, f)
with open('prediction_no_hist_checkpoint_410k.pkl', 'wb') as f:
    pickle.dump(pred_no_hist, f)
print("Đã lưu checkpoint an toàn!")

Đang lưu 644970 users đã xử lý...
✅ Đã lưu checkpoint an toàn!


In [ ]:
print("RESUMING PREDICTION")
try:
    model_xgb.set_params(device='cpu')
except:
    pass 

processed_users = set(pred_with_hist.keys())
all_keys = list(ground_truth_dict.keys())
remaining_keys = [k for k in all_keys if k not in processed_users]

print(f"Tổng user: {len(all_keys)}")
print(f"Đã xử lý: {len(processed_users)}")
print(f"Còn lại: {len(remaining_keys)}")

BATCH_SIZE = 2000
batch_cids_chunked = [remaining_keys[i:i + BATCH_SIZE] for i in range(0, len(remaining_keys), BATCH_SIZE)]

for batch_idx, batch_cids in enumerate(batch_cids_chunked):
    if batch_idx % 5 == 0:
        print(f"   Processing batch {batch_idx * BATCH_SIZE} / {len(remaining_keys)}...")
        gc.collect()

        if batch_idx > 0 and batch_idx % 10 == 0:
            with open('prediction_resume_temp.pkl', 'wb') as f:
                pickle.dump(pred_with_hist, f)

    for cid in batch_cids:
        uid = cust_to_user.get(cid)
        u_idx = user_to_idx.get(uid) if uid else None

        bought_items = set(customer_item_frequency.get(cid, {}).keys())
        candidates = set()
        candidates.update(bought_items)

        if u_idx is not None:
            try:
                ids, _ = model_als.recommend([u_idx], user_items_csr[[u_idx]], N=100, filter_already_liked_items=False)
                for x in ids[0]: candidates.add(idx_to_item[x])
            except: pass

            recent_bought = list(bought_items)[:5]
            for item_b in recent_bought:
                b_idx = item_to_idx.get(item_b)
                if b_idx in item_similarity:
                    for s_idx in item_similarity[b_idx]:
                        candidates.add(idx_to_item[s_idx])

        non_bought_count = sum(1 for c in candidates if c not in bought_items)
        if non_bought_count < 20:
            for top_item in global_top_items[:50]:
                candidates.add(top_item)

        final_cands = list(candidates)
        if not final_cands:
            pred_with_hist[cid] = global_top_items[:10]
            pred_no_hist[cid] = [x for x in global_top_items if x not in bought_items][:10]
            continue

        X_pred = []
        for item_id in final_cands:
            adv = adv_feats_dict.get(uid, {}).get(item_id, {})
            feats = [
                adv.get('cnt', 0), adv.get('cyc', 999), adv.get('ovr', -999), adv.get('last', 999),
                item_popularity.get(item_id, 0),
                global_pop_7d.get(item_id, 0),
                global_pop_30d.get(item_id, 0)
            ]
            X_pred.append(feats)

        X_pred = np.array(X_pred)

        try:
            p1 = model_lgb.predict(X_pred)
            p2 = model_xgb.predict(X_pred)
        except Exception as e:
            p1 = np.zeros(len(X_pred))
            p2 = np.zeros(len(X_pred))

        avg_score = 0.6*p1 + 0.4*p2

        item_scores = list(zip(final_cands, avg_score))
        item_scores.sort(key=lambda x: x[1], reverse=True)

        pred_with_hist[cid] = [x[0] for x in item_scores[:10]]

        no_hist_list = []
        for item, score in item_scores:
            if item not in bought_items:
                no_hist_list.append(item)
                if len(no_hist_list) == 10: break

        if len(no_hist_list) < 10:
            for top in global_top_items:
                if top not in bought_items and top not in no_hist_list:
                    no_hist_list.append(top)
                    if len(no_hist_list) == 10: break
        pred_no_hist[cid] = no_hist_list

print("DONE! Running Evaluation...")

🚀 RESUMING PREDICTION (Safe Mode)
Tổng user: 644970
Đã xử lý: 644970
Còn lại: 0
✅ DONE! Running Evaluation...


In [ ]:
# 1. Hàm tính điểm Precision@K
def precision_at_k(pred_dict, gt_dict, K=10):
    precisions = []
    for user_id, gt_items in gt_dict.items():
        pred_items = pred_dict.get(user_id, [])[:K]

        if not pred_items:
            precisions.append(0)
            continue

        relevant = set(str(x) for x in gt_items)
        pred_set = [str(x) for x in pred_items]

        hits = sum(1 for item in pred_set if item in relevant)
        precisions.append(hits / K)

    return np.mean(precisions) if precisions else 0

# 2. Tính điểm
print("="*40)
print(">>> KẾT QUẢ ĐÁNH GIÁ (EVALUATION)")
print("="*40)

score_with = precision_at_k(pred_with_hist, ground_truth_dict, K=10)
print(f"✅ Score WITH History    : {score_with:.6f}")

# --- Score 2: WITHOUT HISTORY (Chỉ tính món mới) ---
# Kiểm tra biến 'pred_no_hist' (tên đúng trong code chạy loop của bạn)
if 'pred_no_hist' in locals():
    score_no = precision_at_k(pred_no_hist, ground_truth_dict, K=10)
    print(f"✅ Score WITHOUT History : {score_no:.6f}")
else:
    print(" Biến 'pred_no_hist' chưa được tạo.")
print("="*40)

>>> KẾT QUẢ ĐÁNH GIÁ (EVALUATION)
✅ Score WITH History    : 0.085157
✅ Score WITHOUT History : 0.015882


In [ ]:
def save_submission(pred_dict, filename):
    print(f"🔄 Đang chuẩn bị lưu file '{filename}'...")

    formatted_dict = {}
    for uid, items in pred_dict.items():
        str_uid = str(uid)
        str_items = [str(x) for x in items]
        formatted_dict[str_uid] = str_items

    with open(filename, 'wb') as f:
        pickle.dump(formatted_dict, f)

    file_size_mb = os.path.getsize(filename) / (1024 * 1024)
    num_users = len(formatted_dict)

    print("="*40)
    print(f"✅ ĐÃ LƯU THÀNH CÔNG: {filename}")
    print(f"📊 Tổng số User: {num_users:,}")
    print(f"💾 Kích thước file: {file_size_mb:.2f} MB")

    # In thử 1 mẫu để kiểm tra
    first_key = list(formatted_dict.keys())[0]
    print(f"👀 Mẫu dữ liệu (User {first_key}): {formatted_dict[first_key]}")
    print("="*40)

if 'pred_no_hist' in locals():
    save_submission(pred_no_hist, 'prediction_no_hist.pkl')
elif 'prediction_no_hist' in locals():
    save_submission(prediction_no_hist, 'prediction_no_hist.pkl')
else:
    print("Không tìm thấy biến chứa kết quả No-History")

if 'pred_with_hist' in locals():
    save_submission(pred_with_hist, 'prediction_with_hist.pkl')

🔄 Đang chuẩn bị lưu file 'prediction_no_hist.pkl'...
✅ ĐÃ LƯU THÀNH CÔNG: prediction_no_hist.pkl
📊 Tổng số User: 644,970
💾 Kích thước file: 22.41 MB
👀 Mẫu dữ liệu (User 2337685): ['1512000000004', '2024000000010', '0029130000030', '2803000000013', '2808000000001', '2005000000006', '0029120000029', '2803000000011', '2803000000012', '7174000000002']
🔄 Đang chuẩn bị lưu file 'prediction_with_hist.pkl'...
✅ ĐÃ LƯU THÀNH CÔNG: prediction_with_hist.pkl
📊 Tổng số User: 644,970
💾 Kích thước file: 61.53 MB
👀 Mẫu dữ liệu (User 2337685): ['0020010000305', '5137000000002', '1512000000004', '2024000000010', '0029130000030', '2803000000013', '2808000000001', '2005000000006', '0029120000029', '2803000000011']
